# MedSAM2 — DCA1 Augmented Evaluation (Click Prompt)

**Pipeline:**  
1. Augment 134 DCA1 images → ~640 training images (images + masks transformed identically)  
2. 80/20 split on original 134 stems → train on augmented, test on original only  
3. Fine-tune MedSAM2 mask decoder + prompt encoder with centroid click prompt  
4. Evaluate Dice + IoU on 27 held-out original test images

**Bucket:** ` `

## Cell 1 — Environment Setup

In [ ]:
import os, subprocess, sys

subprocess.run(['sudo','apt-get','update','-q'], check=True)
subprocess.run(['sudo','apt-get','install','-y','ffmpeg','p7zip-full'], check=True)
subprocess.run([sys.executable,'-m','pip','install',
    'torch==2.5.1','torchvision==0.20.1',
    '--index-url','https://download.pytorch.org/whl/cu121','-q'], check=True)
subprocess.run([sys.executable,'-m','pip','install',
    'numpy>=2.0.1','opencv-python','pillow','matplotlib','tqdm','scipy','-q'], check=True)

if not os.path.exists(' '):
    subprocess.run(['git','clone','https://github.com/bowang-lab/MedSAM2.git',
                    ' '], check=True)
try:
    import sam2
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-e','.', '-q'],
                   cwd=' ', check=True)

sys.path.insert(0, ' ')
print('Environment ready')

## Cell 2 — Checkpoint + GPU

In [ ]:
import torch, os

BUCKET = ' '
CKPT_DIR   = ' '
CHECKPOINT = os.path.join(CKPT_DIR, 'MedSAM2_latest.pt')
CONFIG     = 'configs/sam2.1_hiera_t512.yaml'
MODEL_SIZE = 512
HIRES      = MODEL_SIZE // 4
BB_FEAT_SIZES = [[HIRES//(2**k)]*2 for k in range(3)]

os.makedirs(CKPT_DIR, exist_ok=True)
if not os.path.exists(CHECKPOINT):
    subprocess.run(['bash','download.sh'], cwd=' ')
else:
    print('Checkpoint present')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} | GPU: {torch.cuda.get_device_name(0)}')

## Cell 3 — Download DCA1

In [ ]:
import glob, numpy as np
from PIL import Image
import matplotlib.pyplot as plt

DCA1_RAW = ' '
os.makedirs(' ', exist_ok=True)
if not os.path.exists(DCA1_RAW):
    os.system('gsutil -m cp -r    ')
else:
    print('DCA1 already on disk')

all_pgm   = glob.glob(f'{DCA1_RAW}/*.pgm')
raw_imgs  = [f for f in all_pgm if '_gt' not in f]
raw_masks = [f for f in all_pgm if '_gt'     in f]

def img_stem(p):  return int(os.path.basename(p).replace('.pgm',''))
def msk_stem(p):  return int(os.path.basename(p).replace('_gt.pgm',''))

imgs_sorted  = sorted(raw_imgs,  key=img_stem)
masks_sorted = sorted(raw_masks, key=msk_stem)

assert all(img_stem(i)==msk_stem(m) for i,m in zip(imgs_sorted,masks_sorted)), 'Pair mismatch!'
print(f'134 pairs verified ✓  (sample: {os.path.basename(imgs_sorted[0])} ↔ {os.path.basename(masks_sorted[0])})')

## Cell 4 — Augment + Process DCA1

**Why augment before splitting?**  
We split on the 134 *original* stems, then augmented versions of training stems are used for training.  
Test set receives **original images only** — no augmented version of a test image is ever in training.  
This avoids data leakage while maximising training variety.

**Augmentations applied (×6 per original image):**

| # | Transform | Why |
|---|-----------|-----|
| 1 | Original | Baseline |
| 2 | Horizontal flip | Mirror-image vessel anatomy |
| 3 | Vertical flip | Inverted projection angles |
| 4 | 90° clockwise rotation | Different catheter entry angles |
| 5 | 45° clockwise rotation + centre-crop | Oblique views common in cath lab |
| 6 | Horizontal flip + 45° rotation | Combines both prior effects |

**Critical:** masks receive the exact same geometric transform as their paired image.  
Images use bilinear interpolation for smooth rotation; masks use nearest-neighbour to preserve hard binary edges.

In [ ]:
from tqdm import tqdm

IMG_DIR  = ' '
MASK_DIR = ' '
os.makedirs(IMG_DIR,  exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)

SUFFIXES = ['orig', 'hf', 'vf', 'r90', 'r45', 'hf_r45']

def rot45_crop(pil, resample, orig_w, orig_h):
    """Rotate 45°, expand canvas, then centre-crop back to original size."""
    rotated = pil.rotate(-45, resample=resample, expand=True)
    rw, rh  = rotated.size
    left    = (rw - orig_w) // 2
    top     = (rh - orig_h) // 2
    return rotated.crop((left, top, left + orig_w, top + orig_h))

def augment_pair(img_pil, msk_pil):
    """Return 6 (img, mask) PIL pairs — image and mask transformed identically."""
    W, H = img_pil.size
    pairs = []

    # 1 — original
    pairs.append((img_pil, msk_pil))

    # 2 — horizontal flip
    pairs.append((
        img_pil.transpose(Image.FLIP_LEFT_RIGHT),
        msk_pil.transpose(Image.FLIP_LEFT_RIGHT)
    ))

    # 3 — vertical flip
    pairs.append((
        img_pil.transpose(Image.FLIP_TOP_BOTTOM),
        msk_pil.transpose(Image.FLIP_TOP_BOTTOM)
    ))

    # 4 — 90° CW rotation
    pairs.append((
        img_pil.rotate(-90, resample=Image.BICUBIC),
        msk_pil.rotate(-90, resample=Image.NEAREST)
    ))

    # 5 — 45° CW rotation + centre-crop
    pairs.append((
        rot45_crop(img_pil, Image.BICUBIC,  W, H),
        rot45_crop(msk_pil, Image.NEAREST, W, H)
    ))

    # 6 — horizontal flip + 45° CW rotation
    img_hf = img_pil.transpose(Image.FLIP_LEFT_RIGHT)
    msk_hf = msk_pil.transpose(Image.FLIP_LEFT_RIGHT)
    pairs.append((
        rot45_crop(img_hf, Image.BICUBIC,  W, H),
        rot45_crop(msk_hf, Image.NEAREST, W, H)
    ))

    return pairs   # len == 6

total_saved = 0
for img_path, msk_path in tqdm(zip(imgs_sorted, masks_sorted), total=134):
    stem    = img_stem(img_path)
    img_pil = Image.open(img_path).convert('L')
    msk_pil = Image.open(msk_path).convert('L')

    # Convert image to RGB (SAM2 encoder expects 3 channels)
    img_rgb = Image.fromarray(np.stack([np.array(img_pil)]*3, axis=-1))

    pairs = augment_pair(img_rgb, msk_pil)

    for suffix, (aug_img, aug_msk) in zip(SUFFIXES, pairs):
        fname = f'{stem}_{suffix}.png'
        aug_img.save(os.path.join(IMG_DIR,  fname))
        # Binarize mask: vessel=255, background=0
        msk_bin = (np.array(aug_msk) > 0).astype(np.uint8) * 255
        Image.fromarray(msk_bin).save(os.path.join(MASK_DIR, fname))
        total_saved += 1

print(f'Saved {total_saved} image+mask pairs  ({total_saved} = 134 × {len(SUFFIXES)})')

## Cell 5 — Verify Augmentations

Visual sanity check: all 6 variants of one image, with mask overlay.  
Confirm that the mask deformation tracks the image deformation exactly.

In [ ]:
stem_check = img_stem(imgs_sorted[0])
fig, axes = plt.subplots(2, 6, figsize=(24, 8))
for col, suffix in enumerate(SUFFIXES):
    img  = np.array(Image.open(f'{IMG_DIR}/{stem_check}_{suffix}.png').convert('L'))
    msk  = np.array(Image.open(f'{MASK_DIR}/{stem_check}_{suffix}.png'))
    axes[0, col].imshow(img,  cmap='gray'); axes[0, col].set_title(suffix); axes[0, col].axis('off')
    axes[1, col].imshow(img,  cmap='gray')
    axes[1, col].imshow(msk > 0, alpha=0.4, cmap='Reds')
    axes[1, col].axis('off')
axes[0, 0].set_ylabel('Image', fontsize=11)
axes[1, 0].set_ylabel('Image + Mask', fontsize=11)
plt.suptitle(f'Augmentation verification — stem {stem_check}', fontsize=13)
plt.tight_layout()
plt.show()

## Cell 6 — Train / Test Split

**Rule:** Split on the 134 original stems (80/20).  
- **Train:** All 6 augmented versions of the 107 training stems → **642 images**  
- **Test:**  Only the `_orig` version of the 27 test stems → **27 images**  

This guarantees no augmented version of a test image ever appears in training.

In [ ]:
import random

all_stems = [img_stem(p) for p in imgs_sorted]
indices   = list(range(134))
random.seed(42)
random.shuffle(indices)

n_train      = int(0.8 * 134)
train_stems  = [all_stems[i] for i in indices[:n_train]]
test_stems   = [all_stems[i] for i in indices[n_train:]]

# Training: all 6 augmented versions of each training stem
train_img_paths  = sorted([f'{IMG_DIR}/{s}_{suf}.png'  for s in train_stems for suf in SUFFIXES])
train_mask_paths = sorted([f'{MASK_DIR}/{s}_{suf}.png' for s in train_stems for suf in SUFFIXES])

# Test: original only
test_img_paths   = sorted([f'{IMG_DIR}/{s}_orig.png'   for s in test_stems])
test_mask_paths  = sorted([f'{MASK_DIR}/{s}_orig.png'  for s in test_stems])

# Verify all files exist
for p in train_img_paths + train_mask_paths + test_img_paths + test_mask_paths:
    assert os.path.exists(p), f'Missing: {p}'

print(f'Train: {len(train_img_paths)} images  ({n_train} stems × {len(SUFFIXES)} augmentations)')
print(f'Test:  {len(test_img_paths)} images  (original only, no augmentation)')

In [ ]:
def centroid_click(mask_np):
    ys, xs = np.where(mask_np > 0)
    if len(ys) == 0:
        return (mask_np.shape[1]//2, mask_np.shape[0]//2)
    return (int(xs.mean()), int(ys.mean()))

def dice_score(pred, gt, smooth=1e-5):
    p, g = pred.astype(float), gt.astype(float)
    return (2*(p*g).sum() + smooth) / (p.sum() + g.sum() + smooth)

def iou_score(pred, gt, smooth=1e-5):
    i = (pred.astype(bool) & gt.astype(bool)).sum()
    u = (pred.astype(bool) | gt.astype(bool)).sum()
    return (i + smooth) / (u + smooth)

print('Metric helpers ready')

## Cell 7 — Zero-Shot Baseline on Test Split

No training. MedSAM2 out of the box. Centroid click on each of the 27 test images.

In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

zs_model     = build_sam2(CONFIG, CHECKPOINT, device=device)
zs_predictor = SAM2ImagePredictor(zs_model)

zs_results = []
for img_path, mask_path in tqdm(zip(test_img_paths, test_mask_paths), total=len(test_img_paths)):
    img_rgb = np.array(Image.open(img_path).convert('RGB'))
    gt_mask = np.array(Image.open(mask_path).convert('L')) > 0
    cx, cy  = centroid_click(gt_mask)
    with torch.inference_mode(), torch.autocast(device_type=device, dtype=torch.bfloat16):
        zs_predictor.set_image(img_rgb)
        preds, _, _ = zs_predictor.predict(
            point_coords=np.array([[cx, cy]]), point_labels=np.array([1]),
            multimask_output=False)
    pred_mask = preds[0]
    zs_results.append({'stem': int(os.path.splitext(os.path.basename(img_path))[0].split('_')[0]),
                       'dice': dice_score(pred_mask, gt_mask),
                       'iou':  iou_score(pred_mask,  gt_mask),
                       'pred': pred_mask, 'gt': gt_mask, 'img': img_rgb})

zs_dice = [r['dice'] for r in zs_results]
zs_iou  = [r['iou']  for r in zs_results]
print(f'Zero-Shot | Dice: {np.mean(zs_dice):.3f} ± {np.std(zs_dice):.3f} | IoU: {np.mean(zs_iou):.3f} ± {np.std(zs_iou):.3f}')

## Cell 8 — Dataset + Fine-Tuning

**Augmentation during training:** None needed — augmented images are already on disk.  
The Dataset simply loads the pre-generated files.  
This also means augmentation is deterministic and reproducible (not random per-epoch).

**pos_weight:** Computed from training masks — tells BCE how much to upweight vessel pixels relative to background.

In [ ]:
import torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import GradScaler

class DCA1Dataset(Dataset):
    def __init__(self, img_paths, mask_paths):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.transform  = transforms.Compose([
            transforms.Resize((MODEL_SIZE, MODEL_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
        ])
    def __len__(self): return len(self.img_paths)
    def __getitem__(self, idx):
        img      = Image.open(self.img_paths[idx]).convert('RGB')
        mask_pil = Image.open(self.mask_paths[idx]).convert('L')
        mask_np  = np.array(mask_pil)
        img_t    = self.transform(img)
        mask_256 = np.array(mask_pil.resize((256,256), Image.NEAREST))
        mask_t   = torch.from_numpy((mask_256 > 0).astype(np.float32)).unsqueeze(0)
        ys, xs   = np.where(mask_np > 0)
        if len(ys) > 0:
            pick = np.random.randint(len(ys))
            py, px = int(ys[pick]), int(xs[pick])
        else:
            py, px = mask_np.shape[0]//2, mask_np.shape[1]//2
        h, w  = mask_np.shape
        point = torch.tensor([px/w*MODEL_SIZE, py/h*MODEL_SIZE], dtype=torch.float32)
        return img_t, mask_t, point

# pos_weight from training masks
fg_list = []
for mp in train_mask_paths:
    m = np.array(Image.open(mp).convert('L').resize((256,256), Image.NEAREST))
    fg_list.append((m > 0).mean())
fg_mean    = np.mean(fg_list)
pos_weight = torch.tensor([(1-fg_mean)/fg_mean], device=device)
print(f'Foreground fraction: {fg_mean*100:.1f}%  |  pos_weight: {pos_weight.item():.1f}x')

train_ds     = DCA1Dataset(train_img_paths, train_mask_paths)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} images | {len(train_loader)} batches/epoch')

## Cell 9 — Load Model + Freeze Encoder

In [ ]:
model = build_sam2(CONFIG, CHECKPOINT, device=device)

for name, param in model.named_parameters():
    param.requires_grad = ('image_encoder' not in name)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)  — mask decoder + prompt encoder')

## Cell 10 — Training Loop

**What's happening inside each step:**  
1. Load a batch of 4 augmented images + their masks + click points  
2. Frozen encoder → feature maps (no gradient here)  
3. Prompt encoder converts the click → sparse embeddings  
4. Mask decoder combines features + embeddings → 256×256 logit map  
5. Dice loss + weighted BCE loss → backprop → update decoder + prompt encoder weights  
6. Gradient clipping at 1.0 prevents exploding gradients

In [ ]:
CKPT_PATH = ' '
EPOCHS    = 20

def dice_loss(logits, target, smooth=1e-5):
    pred  = torch.sigmoid(logits)
    inter = (pred * target).sum(dim=(2,3))
    union = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
    return (1 - (2*inter + smooth)/(union + smooth)).mean()

def combined_loss(logits, target, pw):
    return dice_loss(logits, target) + nn.BCEWithLogitsLoss(pos_weight=pw)(logits, target)

optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = GradScaler()
best_loss = float('inf')
train_losses = []

print(f'Training {EPOCHS} epochs on {len(train_ds)} augmented images...')

for epoch in range(EPOCHS):
    model.train()
    ep_loss = 0.0
    for imgs, masks, points in tqdm(train_loader, desc=f'Ep {epoch+1}/{EPOCHS}', leave=False):
        optimizer.zero_grad()
        B = imgs.shape[0]
        imgs_dev, points_dev = imgs.to(device), points.to(device)

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            with torch.no_grad():
                backbone_out = model.forward_image(imgs_dev)
                _, vision_feats, _, _ = model._prepare_backbone_features(backbone_out)
                if model.directly_add_no_mem_embed:
                    vision_feats[-1] = vision_feats[-1] + model.no_mem_embed
            feats = [feat.permute(1,2,0).view(B,-1,*fs)
                     for feat, fs in zip(vision_feats[::-1], BB_FEAT_SIZES[::-1])][::-1]
            image_embed, high_res_feats = feats[-1], feats[:-1]

            all_logits = []
            for i in range(B):
                pt       = points_dev[i].reshape(1,1,2)
                pt_label = torch.ones(1,1, dtype=torch.int, device=device)
                sparse_emb, dense_emb = model.sam_prompt_encoder(
                    points=(pt, pt_label), boxes=None, masks=None)
                low_res_masks, _, _, _ = model.sam_mask_decoder(
                    image_embeddings=image_embed[i].unsqueeze(0),
                    image_pe=model.sam_prompt_encoder.get_dense_pe(),
                    sparse_prompt_embeddings=sparse_emb,
                    dense_prompt_embeddings=dense_emb,
                    multimask_output=False, repeat_image=False,
                    high_res_features=[f[i].unsqueeze(0) for f in high_res_feats])
                all_logits.append(low_res_masks)

            logits  = torch.cat(all_logits, dim=0)
            targets = masks.to(device)
            if targets.shape[-2:] != logits.shape[-2:]:
                targets = F.interpolate(targets.float(), size=logits.shape[-2:], mode='nearest')
            loss = combined_loss(logits, targets, pos_weight)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()
        ep_loss += loss.item()

    ep_loss /= len(train_loader)
    train_losses.append(ep_loss)
    scheduler.step()
    print(f'Epoch {epoch+1:2d} | Loss: {ep_loss:.4f}')
    if ep_loss < best_loss:
        best_loss = ep_loss
        torch.save(model.state_dict(), CKPT_PATH)
        print(f'           ★ saved (loss {best_loss:.4f})')

print(f'\nDone. Best loss: {best_loss:.4f}')
!gsutil cp {CKPT_PATH} {BUCKET}/checkpoints/medsam2_dca1_aug.pt

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(range(1,EPOCHS+1), train_losses, 'b-o', markersize=4)
plt.xlabel('Epoch'); plt.ylabel('Loss (Dice + wBCE)')
plt.title(f'Training Loss — 642 augmented images, {EPOCHS} epochs')
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(' ', dpi=100); plt.show()
!gsutil cp   {BUCKET}/results/aug_loss_curve.png

## Cell 11 — Fine-Tuned Inference on Test Split

Load the best checkpoint, verify nothing is missing, run on the 27 original test images.

In [ ]:
ft_model = build_sam2(CONFIG, CHECKPOINT, device=device)
incomp   = ft_model.load_state_dict(torch.load(CKPT_PATH, map_location=device), strict=False)
print(f'Missing: {len(incomp.missing_keys)} | Unexpected: {len(incomp.unexpected_keys)}')
assert len(incomp.missing_keys) == 0, '⚠ Checkpoint load failed — missing keys!'
print('✓ Clean checkpoint load')
ft_model.eval()
ft_predictor = SAM2ImagePredictor(ft_model)

ft_results = []
for img_path, mask_path in tqdm(zip(test_img_paths, test_mask_paths), total=len(test_img_paths)):
    img_rgb = np.array(Image.open(img_path).convert('RGB'))
    gt_mask = np.array(Image.open(mask_path).convert('L')) > 0
    cx, cy  = centroid_click(gt_mask)
    with torch.inference_mode(), torch.autocast(device_type=device, dtype=torch.bfloat16):
        ft_predictor.set_image(img_rgb)
        preds, _, _ = ft_predictor.predict(
            point_coords=np.array([[cx, cy]]), point_labels=np.array([1]),
            multimask_output=False)
    pred_mask = preds[0]
    ft_results.append({'stem': int(os.path.splitext(os.path.basename(img_path))[0].split('_')[0]),
                       'dice': dice_score(pred_mask, gt_mask),
                       'iou':  iou_score(pred_mask,  gt_mask),
                       'pred': pred_mask, 'gt': gt_mask, 'img': img_rgb})

ft_dice = [r['dice'] for r in ft_results]
ft_iou  = [r['iou']  for r in ft_results]
print(f'\n=== Results (27 test images, original only) ===')
print(f'Zero-Shot             | Dice: {np.mean(zs_dice):.3f} ± {np.std(zs_dice):.3f} | IoU: {np.mean(zs_iou):.3f}')
print(f'Fine-Tuned (aug, 20ep)| Dice: {np.mean(ft_dice):.3f} ± {np.std(ft_dice):.3f} | IoU: {np.mean(ft_iou):.3f}')

## Cell 12 — Results

In [ ]:
# Per-image table
print(f"{'Stem':>6} | {'ZS Dice':>8} | {'FT Dice':>8} | {'Delta':>7}")
print('-'*40)
for zr, fr in zip(sorted(zs_results, key=lambda r: r['stem']),
                  sorted(ft_results,  key=lambda r: r['stem'])):
    delta = fr['dice'] - zr['dice']
    mark  = '↑' if delta > 0.01 else ('↓' if delta < -0.01 else '~')
    print(f"{zr['stem']:>6} | {zr['dice']:>8.3f} | {fr['dice']:>8.3f} | {delta:>+6.3f} {mark}")
print(f"{'MEAN':>6} | {np.mean(zs_dice):>8.3f} | {np.mean(ft_dice):>8.3f} | {np.mean(ft_dice)-np.mean(zs_dice):>+6.3f}")

In [ ]:
# Visual panel — worst / mid / best fine-tuned Dice
ft_sorted = sorted(ft_results, key=lambda r: r['dice'])
zs_by_stem = {r['stem']: r for r in zs_results}
show = [ft_sorted[0], ft_sorted[len(ft_sorted)//3], ft_sorted[2*len(ft_sorted)//3],
        ft_sorted[-3], ft_sorted[-2], ft_sorted[-1]]

fig, axes = plt.subplots(3, 6, figsize=(26, 11))
for col, fr in enumerate(show):
    zr  = zs_by_stem[fr['stem']]
    img = fr['img'][:,:,0]
    axes[0,col].imshow(img, cmap='gray')
    axes[0,col].imshow(fr['gt'], alpha=0.35, cmap='Greens')
    axes[0,col].set_title(f"#{fr['stem']}"); axes[0,col].axis('off')
    axes[1,col].imshow(img, cmap='gray')
    axes[1,col].imshow(zr['pred'], alpha=0.45, cmap='Reds')
    axes[1,col].set_title(f"ZS D={zr['dice']:.2f}"); axes[1,col].axis('off')
    axes[2,col].imshow(img, cmap='gray')
    axes[2,col].imshow(fr['pred'], alpha=0.45, cmap='Blues')
    axes[2,col].set_title(f"FT D={fr['dice']:.2f}"); axes[2,col].axis('off')

for row, label in enumerate(['GT overlay','Zero-Shot (Red)','Fine-Tuned aug (Blue)']):
    axes[row,0].set_ylabel(label, fontsize=10)
plt.suptitle(f'ZS Dice={np.mean(zs_dice):.3f}  →  FT Dice={np.mean(ft_dice):.3f}  (642 aug training images, 20ep)', fontsize=12)
plt.tight_layout()
plt.savefig(' ', dpi=120, bbox_inches='tight')
plt.show()
!gsutil cp   {BUCKET}/results/dca1_aug_comparison.png
print('Results saved to GCS')